# 25 — ICH 3D Residual U-Net v2 with Expanded True Negatives

## Goal

A stronger and fairer 3D follow-up to Notebook 19.

Changes versus the first 3D prototype:
- higher in-plane resolution: 320 instead of 256,
- larger residual 3D U-Net,
- more patch exposure per epoch,
- more epochs,
- loss weighted more toward cross-entropy to suppress false-positive voxels,
- same expanded true-negative strategy,
- same common TRAIN / DEV split.

This is still a controlled 3D experiment; the locked TEST split is not used.


## 1. Environment

In [ ]:
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg

## 2. Imports

In [ ]:
import json
import random
import time
import warnings
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from pydicom.pixels import apply_modality_lut
from sklearn.metrics import confusion_matrix, f1_score, mean_absolute_error, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Invalid value for VR UI.*")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## 3. Reproducibility and paths

In [ ]:
SEED = 20260918
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TEST = 68
N_DEV = 54
N_SPLIT_TRIALS = 1500

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/mehdipaykanheyrati/iaaa-contest-bct"),
    Path("/kaggle/input/iaaa-contest-bct"),
]

OUTPUT_ROOT = Path("/kaggle/working/ich_3d_resunet_v2_expanded_true_negatives")
MODELS_DIR = OUTPUT_ROOT / "models"
METRICS_DIR = OUTPUT_ROOT / "metrics"
CACHE_DIR = OUTPUT_ROOT / "cache"
SPLITS_DIR = OUTPUT_ROOT / "splits"
for path in [MODELS_DIR, METRICS_DIR, CACHE_DIR, SPLITS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Output root:", OUTPUT_ROOT)

## 4. 3D experiment configuration

In [ ]:
ICH_CLASSES = ["EDH", "SDH", "IPH", "SAH", "IVH"]
ICH_COLUMNS = [f"V_{name}" for name in ICH_CLASSES]
ORIGINAL_CLASS_ID = {"EDH": 4, "SDH": 3, "IPH": 2, "SAH": 5, "IVH": 1}

BLOOD_WINDOW_CENTER = 45.0
BLOOD_WINDOW_WIDTH = 100.0

PATCH_DEPTH = 16
CENTER_SLOT = PATCH_DEPTH // 2
IMAGE_SIZE = 320
INFERENCE_STRIDE = 8
BASE_CHANNELS = 24

BATCH_SIZE = 1
NUM_WORKERS = 2
EPOCHS = 30
SAMPLES_PER_EPOCH = 1800
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 8
HU_CACHE_SIZE = 96

CATEGORY_WEIGHT = {
    "positive_slice": 1.0,
    "negative_in_positive_series": 1.0,
    "expanded_negative_in_normal_series": 1.0,
}

THRESHOLD_GRID = np.arange(0.20, 0.96, 0.05)
FIXED_CONTINUITY_MIN_RUN = 3
CUSTOM_INIT_CHECKPOINT = None

print("Patch:", (PATCH_DEPTH, IMAGE_SIZE, IMAGE_SIZE))
print("Base channels:", BASE_CHANNELS)
print("Samples per epoch:", SAMPLES_PER_EPOCH)

### Optional 3D pretraining

This notebook uses a custom PyTorch 3D U-Net so the final model does not depend on MONAI.

To initialize from a compatible 3D checkpoint, set:

```python
CUSTOM_INIT_CHECKPOINT = "/kaggle/input/.../checkpoint.pth"
```

Only matching tensor names and shapes are loaded. The default is training from scratch.

## 5. Locate the CT data

In [ ]:
def first_existing(paths):
    return next((path for path in paths if path.exists()), None)

def normalize_series_id(value):
    if pd.isna(value):
        return ""
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()

DATASET_ROOT = first_existing(DATASET_ROOT_CANDIDATES)
if DATASET_ROOT is None:
    raise FileNotFoundError("CT dataset root was not found.")

DATA_ROOT = first_existing([DATASET_ROOT / "iaaa-contest-bct" / "Data", DATASET_ROOT / "Data"])
if DATA_ROOT is None:
    raise FileNotFoundError("Data directory was not found.")

TRAINING_DIR = DATA_ROOT / "training"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
if not TRAINING_DIR.exists() or not ANNOTATIONS_DIR.exists():
    raise FileNotFoundError("Data/training or Data/annotations was not found.")

TARGETS_PATH = first_existing([
    DATASET_ROOT / "series_targets_df.csv",
    DATASET_ROOT / "iaaa-contest-bct" / "series_targets_df.csv",
    DATA_ROOT / "series_targets_df.csv",
    DATA_ROOT.parent / "series_targets_df.csv",
])
if TARGETS_PATH is None:
    raise FileNotFoundError("series_targets_df.csv was not found.")

print("Dataset root:", DATASET_ROOT)
print("Targets:", TARGETS_PATH)

## 6. Reconstruct the common TRAIN / DEV split

In [ ]:
targets_df = pd.read_csv(TARGETS_PATH).drop(columns=["Unnamed: 0"], errors="ignore").copy()
REQUIRED_TARGET_COLUMNS = ["series_id", "V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm", "triage_class"]

targets_df["series_id"] = targets_df["series_id"].map(normalize_series_id)
for column in ["V_EDH", "V_SDH", "V_IPH", "V_SAH", "V_IVH", "fracture_prob", "MLS_mm"]:
    targets_df[column] = pd.to_numeric(targets_df[column], errors="raise")
targets_df["triage_class"] = pd.to_numeric(targets_df["triage_class"], errors="raise").astype(int)
targets_df = targets_df.drop_duplicates("series_id").reset_index(drop=True)

if len(targets_df) != 338:
    raise RuntimeError(f"Expected 338 target series, found {len(targets_df)}.")

def add_split_features(dataframe):
    result = dataframe.copy()
    total_ich = result[ICH_COLUMNS].sum(axis=1)
    result["feature_any_ich"] = (total_ich >= 0.1).astype(int)
    result["feature_fracture"] = (result["fracture_prob"] >= 0.5).astype(int)
    for column in ICH_COLUMNS:
        result[f"feature_{column}"] = (result[column] >= 0.1).astype(int)
    result["mls_bin"] = pd.cut(result["MLS_mm"], bins=[-0.01, 1.0, 3.0, 5.0, np.inf], labels=False, include_lowest=True).astype(int)
    for triage_class in [0, 1, 2]:
        result[f"feature_triage_{triage_class}"] = (result["triage_class"] == triage_class).astype(int)
    for mls_bin in [0, 1, 2, 3]:
        result[f"feature_mls_bin_{mls_bin}"] = (result["mls_bin"] == mls_bin).astype(int)
    return result

BALANCE_COLUMNS = [
    "feature_triage_0", "feature_triage_1", "feature_triage_2", "feature_any_ich", "feature_fracture",
    "feature_V_EDH", "feature_V_SDH", "feature_V_IPH", "feature_V_SAH", "feature_V_IVH",
    "feature_mls_bin_0", "feature_mls_bin_1", "feature_mls_bin_2", "feature_mls_bin_3",
]

def choose_balanced_subset(dataframe, subset_size, seed_start, n_trials):
    best = None
    full_prevalence = dataframe[BALANCE_COLUMNS].mean()
    required = ["feature_fracture", "feature_V_EDH", "feature_V_SDH", "feature_V_IPH", "feature_V_SAH", "feature_V_IVH", "feature_mls_bin_1", "feature_mls_bin_2", "feature_mls_bin_3"]

    for trial_seed in range(seed_start, seed_start + n_trials):
        remaining, subset = train_test_split(dataframe, test_size=int(subset_size), random_state=int(trial_seed), shuffle=True, stratify=dataframe["triage_class"])
        if any(subset[column].sum() == 0 for column in required) or any(remaining[column].sum() == 0 for column in required):
            continue
        score = float((subset[BALANCE_COLUMNS].mean() - full_prevalence).abs().mean() + (remaining[BALANCE_COLUMNS].mean() - full_prevalence).abs().mean())
        if best is None or score < best["score"]:
            best = {"score": score, "seed": trial_seed, "remaining": remaining.copy(), "subset": subset.copy()}

    if best is None:
        raise RuntimeError("Could not reconstruct the common split.")
    return best

split_df = add_split_features(targets_df)
test_split = choose_balanced_subset(split_df, N_TEST, SEED, N_SPLIT_TRIALS)
train_dev_df = test_split["remaining"].copy().reset_index(drop=True)
test_df = test_split["subset"].copy().reset_index(drop=True)
dev_split = choose_balanced_subset(train_dev_df, N_DEV, SEED + N_SPLIT_TRIALS + 1, N_SPLIT_TRIALS)
train_series_df = dev_split["remaining"][REQUIRED_TARGET_COLUMNS].copy().reset_index(drop=True)
dev_series_df = dev_split["subset"][REQUIRED_TARGET_COLUMNS].copy().reset_index(drop=True)

TRAIN_SERIES = set(train_series_df["series_id"])
DEV_SERIES = set(dev_series_df["series_id"])
TEST_SERIES = set(test_df["series_id"])

if TRAIN_SERIES & DEV_SERIES or TRAIN_SERIES & TEST_SERIES or DEV_SERIES & TEST_SERIES:
    raise RuntimeError("Series overlap detected.")
if len(TRAIN_SERIES) != 216 or len(DEV_SERIES) != 54 or len(TEST_SERIES) != 68:
    raise RuntimeError("Unexpected split sizes.")

train_series_df.to_csv(SPLITS_DIR / "common_train_series.csv", index=False)
dev_series_df.to_csv(SPLITS_DIR / "common_dev_series.csv", index=False)

print("TRAIN:", len(TRAIN_SERIES), "| DEV:", len(DEV_SERIES), "| held-out TEST:", len(TEST_SERIES))

## 7. Build the master slice index

In [ ]:
def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return float(default)

def safe_tuple(value):
    try:
        return tuple(float(item) for item in value)
    except Exception:
        return None

def parse_rle_header(rle):
    if not isinstance(rle, dict) or "shape" not in rle or "counts" not in rle:
        return None
    shape = tuple(int(value) for value in rle["shape"])
    counts = [int(value) for value in rle["counts"]]
    if len(shape) != 2 or len(counts) % 2 != 0 or sum(counts[1::2]) != int(np.prod(shape)):
        return None
    return shape, counts

def read_dicom_header(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
    spacing = safe_tuple(getattr(ds, "PixelSpacing", None))
    orientation = safe_tuple(getattr(ds, "ImageOrientationPatient", None))
    position = safe_tuple(getattr(ds, "ImagePositionPatient", None))
    position_scalar = np.nan

    if orientation is not None and position is not None and len(orientation) >= 6 and len(position) >= 3:
        row_cosine = np.asarray(orientation[:3], dtype=np.float64)
        col_cosine = np.asarray(orientation[3:6], dtype=np.float64)
        normal = np.cross(row_cosine, col_cosine)
        position_scalar = float(np.dot(np.asarray(position[:3], dtype=np.float64), normal))

    return {
        "sop_uid": str(getattr(ds, "SOPInstanceUID", path.stem)),
        "rows": int(getattr(ds, "Rows", 512)),
        "cols": int(getattr(ds, "Columns", 512)),
        "row_spacing": spacing[0] if spacing is not None else np.nan,
        "col_spacing": spacing[1] if spacing is not None else np.nan,
        "instance_number": safe_float(getattr(ds, "InstanceNumber", np.nan)),
        "position_scalar": position_scalar,
        "slice_thickness": abs(safe_float(getattr(ds, "SliceThickness", np.nan))),
        "spacing_between_slices": abs(safe_float(getattr(ds, "SpacingBetweenSlices", np.nan))),
    }

MASTER_INDEX_PATH = CACHE_DIR / "master_slice_index.pkl"

if MASTER_INDEX_PATH.exists():
    master_df = pd.read_pickle(MASTER_INDEX_PATH)
else:
    rows = []
    series_dirs = sorted([path for path in TRAINING_DIR.iterdir() if path.is_dir()], key=lambda path: int(path.name))

    for series_dir in tqdm(series_dirs, desc="Building master slice index"):
        series_id = normalize_series_id(series_dir.name)
        split_name = "TRAIN" if series_id in TRAIN_SERIES else "DEV" if series_id in DEV_SERIES else "TEST"

        for dicom_path in sorted(series_dir.glob("*.dcm")):
            header = read_dicom_header(dicom_path)
            annotation_path = ANNOTATIONS_DIR / series_id / f"{header['sop_uid']}.json"
            segmentation_rle = None
            segmentation_valid = 0
            n_positive_ich_classes = np.nan

            if annotation_path.exists():
                with open(annotation_path, "r", encoding="utf-8") as file:
                    annotation = json.load(file)
                parsed = parse_rle_header(annotation.get("segmentation_rle"))
                if parsed is not None:
                    _, counts = parsed
                    labels_present = set(counts[0::2])
                    n_positive_ich_classes = len(labels_present & {1, 2, 3, 4, 5})
                    segmentation_rle = json.dumps(annotation["segmentation_rle"], separators=(",", ":"))
                    segmentation_valid = 1

            rows.append({
                "series_id": series_id,
                "split": split_name,
                "dicom_path": str(dicom_path),
                "sop_uid": header["sop_uid"],
                "rows": header["rows"],
                "cols": header["cols"],
                "row_spacing": header["row_spacing"],
                "col_spacing": header["col_spacing"],
                "instance_number": header["instance_number"],
                "position_scalar": header["position_scalar"],
                "slice_thickness": header["slice_thickness"],
                "spacing_between_slices": header["spacing_between_slices"],
                "segmentation_rle": segmentation_rle,
                "segmentation_valid": segmentation_valid,
                "n_positive_ich_classes": n_positive_ich_classes,
            })

    master_df = pd.DataFrame(rows)
    master_df.to_pickle(MASTER_INDEX_PATH)

master_df["series_id"] = master_df["series_id"].map(normalize_series_id)

print("Master slices:", len(master_df))
print("Valid segmentation slices:", int(master_df["segmentation_valid"].sum()))

## 8. Build expanded-negative center candidates and ordered series

In [ ]:
train_targets = train_series_df[["series_id"] + ICH_COLUMNS].copy()
dev_targets = dev_series_df[["series_id"] + ICH_COLUMNS].copy()
train_targets["series_true_any_ich"] = (train_targets[ICH_COLUMNS].sum(axis=1) >= 0.1).astype(int)
dev_targets["series_true_any_ich"] = (dev_targets[ICH_COLUMNS].sum(axis=1) >= 0.1).astype(int)

master_df = master_df.merge(pd.concat([
    train_targets[["series_id", "series_true_any_ich"]],
    dev_targets[["series_id", "series_true_any_ich"]],
], ignore_index=True), on="series_id", how="left")

train_all = master_df[master_df["split"] == "TRAIN"].copy()
dev_all = master_df[master_df["split"] == "DEV"].copy()

train_positive_series_labeled = train_all[(train_all["series_true_any_ich"] == 1) & (train_all["segmentation_valid"] == 1)].copy()
train_normal_series_all = train_all[train_all["series_true_any_ich"] == 0].copy()
train_centers = pd.concat([train_positive_series_labeled, train_normal_series_all], ignore_index=True)
dev_centers = dev_all[dev_all["segmentation_valid"] == 1].copy()

train_centers["slice_has_ich"] = (train_centers["n_positive_ich_classes"].fillna(0) > 0).astype(int)
train_centers["sample_category"] = np.select(
    [
        train_centers["slice_has_ich"] == 1,
        (train_centers["series_true_any_ich"] == 1) & (train_centers["slice_has_ich"] == 0),
        train_centers["series_true_any_ich"] == 0,
    ],
    ["positive_slice", "negative_in_positive_series", "expanded_negative_in_normal_series"],
    default="unknown",
)

def sort_series_rows(dataframe):
    groups = {}
    lookup = {}

    for series_id, group in dataframe.groupby("series_id", sort=False):
        group = group.copy()
        if group["position_scalar"].notna().all():
            group = group.sort_values("position_scalar")
        elif group["instance_number"].notna().all():
            group = group.sort_values("instance_number")
        else:
            group = group.sort_values("dicom_path")
        group = group.reset_index(drop=True)
        groups[series_id] = group

        for index, row in group.iterrows():
            lookup[(series_id, str(row["sop_uid"]))] = index

    return groups, lookup

series_rows, center_lookup = sort_series_rows(master_df)

audit = train_centers.groupby("sample_category").size().reset_index(name="n_center_candidates")
audit.to_csv(METRICS_DIR / "01_training_center_audit.csv", index=False)
display(audit)
print("TRAIN center candidates:", len(train_centers))
print("DEV labeled centers:", len(dev_centers))

## 9. Image, mask, and patch preprocessing

In [ ]:
@lru_cache(maxsize=HU_CACHE_SIZE)
def load_hu_image(path):
    ds = pydicom.dcmread(path, force=True)
    return np.asarray(apply_modality_lut(ds.pixel_array, ds), dtype=np.float32)

def parse_rle(rle):
    if not isinstance(rle, dict) or "shape" not in rle or "counts" not in rle:
        return None
    shape = tuple(int(value) for value in rle["shape"])
    counts = [int(value) for value in rle["counts"]]
    if len(shape) != 2 or len(counts) % 2 != 0 or sum(counts[1::2]) != int(np.prod(shape)):
        return None
    return shape, counts

def decode_ich_mask(rle_json):
    shape, counts = parse_rle(json.loads(rle_json))
    original = np.empty(int(np.prod(shape)), dtype=np.uint8)
    position = 0

    for value, length in zip(counts[0::2], counts[1::2]):
        original[position:position + length] = value
        position += length

    original = original.reshape(shape)
    mask = np.zeros(shape, dtype=np.uint8)

    for new_id, subtype in enumerate(ICH_CLASSES, start=1):
        mask[original == ORIGINAL_CLASS_ID[subtype]] = new_id

    return torch.from_numpy(mask).long()

def blood_window_tensor(image_hu):
    low = BLOOD_WINDOW_CENTER - BLOOD_WINDOW_WIDTH / 2.0
    high = BLOOD_WINDOW_CENTER + BLOOD_WINDOW_WIDTH / 2.0
    image = torch.from_numpy(((np.clip(image_hu, low, high) - low) / (high - low)).astype(np.float32))
    return F.interpolate(image[None, None], size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)[0, 0]

def resize_mask(mask):
    return F.interpolate(mask[None, None].float(), size=(IMAGE_SIZE, IMAGE_SIZE), mode="nearest")[0, 0].long()

def patch_indices(center_index, n_slices):
    start = center_index - CENTER_SLOT
    raw = [start + offset for offset in range(PATCH_DEPTH)]
    return [min(max(index, 0), n_slices - 1) for index in raw]

## 10. 3D patch dataset with masked supervision

In [ ]:
class ICH3DPatchDataset(Dataset):
    def __init__(self, centers, training=False):
        self.centers = centers.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, index):
        center_row = self.centers.iloc[index]
        series_id = center_row["series_id"]
        group = series_rows[series_id]
        center_index = center_lookup[(series_id, str(center_row["sop_uid"]))]
        indices = patch_indices(center_index, len(group))

        images = []
        targets = []
        supervised = []

        for slice_index in indices:
            row = group.iloc[slice_index]
            images.append(blood_window_tensor(load_hu_image(str(row["dicom_path"]))))

            if int(row["series_true_any_ich"]) == 0:
                targets.append(torch.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=torch.long))
                supervised.append(1.0)
            elif int(row["segmentation_valid"]) == 1:
                targets.append(resize_mask(decode_ich_mask(row["segmentation_rle"])))
                supervised.append(1.0)
            else:
                targets.append(torch.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=torch.long))
                supervised.append(0.0)

        volume = torch.stack(images).unsqueeze(0)
        target = torch.stack(targets)
        supervision = torch.tensor(supervised, dtype=torch.float32)

        if self.training and random.random() < 0.5:
            volume = torch.flip(volume, dims=[-1])
            target = torch.flip(target, dims=[-1])

        if self.training and random.random() < 0.15:
            volume = torch.flip(volume, dims=[1])
            target = torch.flip(target, dims=[0])
            supervision = torch.flip(supervision, dims=[0])

        return volume, target, supervision, series_id, str(center_row["sop_uid"])

## 11. Series-balanced patch sampling

In [ ]:
def build_sampler(dataframe):
    series_counts = dataframe.groupby("series_id").size().to_dict()
    weights = []

    for row in dataframe.itertuples(index=False):
        weights.append((1.0 / series_counts[row.series_id]) * CATEGORY_WEIGHT[row.sample_category])

    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=SAMPLES_PER_EPOCH, replacement=True)

train_dataset = ICH3DPatchDataset(train_centers, training=True)
dev_dataset = ICH3DPatchDataset(dev_centers, training=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=build_sampler(train_centers), num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
dev_loader = DataLoader(dev_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)

print("TRAIN patches sampled per epoch:", SAMPLES_PER_EPOCH)
print("DEV center patches:", len(dev_dataset))

## 12. Custom 3D U-Net

In [ ]:
def norm3d(channels):
    groups = 8 if channels >= 8 else 1
    return nn.GroupNorm(groups, channels)

class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False)
        self.norm1 = norm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = norm3d(out_channels)
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv3d(in_channels, out_channels, 1, bias=False)

    def forward(self, x):
        residual = self.skip(x)
        x = F.silu(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return F.silu(x + residual)

class ResUNet3D(nn.Module):
    def __init__(self, in_channels=1, num_classes=6, base=24):
        super().__init__()
        self.enc1 = ResidualBlock3D(in_channels, base)
        self.enc2 = ResidualBlock3D(base, base * 2)
        self.enc3 = ResidualBlock3D(base * 2, base * 4)
        self.bottleneck = ResidualBlock3D(base * 4, base * 8)
        self.pool = nn.MaxPool3d(2)
        self.up3 = nn.ConvTranspose3d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ResidualBlock3D(base * 8, base * 4)
        self.up2 = nn.ConvTranspose3d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ResidualBlock3D(base * 4, base * 2)
        self.up1 = nn.ConvTranspose3d(base * 2, base, 2, stride=2)
        self.dec1 = ResidualBlock3D(base * 2, base)
        self.out = nn.Conv3d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

def load_matching_checkpoint(model, path):
    checkpoint = torch.load(path, map_location="cpu")
    state = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint)) if isinstance(checkpoint, dict) else checkpoint
    target = model.state_dict()
    matching = {key: value for key, value in state.items() if key in target and target[key].shape == value.shape}
    target.update(matching)
    model.load_state_dict(target)
    print(f"Transferred {len(matching)}/{len(target)} matching tensors.")

model = ResUNet3D(base=BASE_CHANNELS).to(DEVICE)

if CUSTOM_INIT_CHECKPOINT is not None:
    checkpoint_path = Path(CUSTOM_INIT_CHECKPOINT)
    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)
    load_matching_checkpoint(model, checkpoint_path)

print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


## 13. Masked CE + Dice loss

In [ ]:
def masked_loss(logits, targets, supervision):
    voxel_mask = supervision[:, :, None, None]
    ce_map = F.cross_entropy(logits, targets, reduction="none")
    ce_denominator = voxel_mask.sum() * targets.shape[-2] * targets.shape[-1]
    ce = (ce_map * voxel_mask).sum() / ce_denominator.clamp_min(1.0)

    probabilities = torch.softmax(logits, dim=1)
    one_hot = F.one_hot(targets, num_classes=6).permute(0, 4, 1, 2, 3).float()
    supervised_voxels = supervision[:, None, :, None, None]
    dice_losses = []

    for class_id in range(1, 6):
        pred = probabilities[:, class_id:class_id + 1] * supervised_voxels
        true = one_hot[:, class_id:class_id + 1] * supervised_voxels
        numerator = 2.0 * (pred * true).sum() + 1e-5
        denominator = pred.sum() + true.sum() + 1e-5
        dice_losses.append(1.0 - numerator / denominator)

    dice = torch.stack(dice_losses).mean()
    return 0.7 * ce + 0.3 * dice, ce.detach(), dice.detach()

@torch.inference_mode()
def evaluate_center_dice(model, loader):
    model.eval()
    intersections = np.zeros(5, dtype=np.float64)
    denominators = np.zeros(5, dtype=np.float64)

    for volumes, targets, supervision, _, _ in loader:
        volumes = volumes.to(DEVICE, non_blocking=True)
        predictions = torch.argmax(model(volumes)[:, :, CENTER_SLOT], dim=1).cpu()
        truth = targets[:, CENTER_SLOT]

        for class_id in range(1, 6):
            pred = predictions == class_id
            true = truth == class_id
            intersections[class_id - 1] += float((pred & true).sum())
            denominators[class_id - 1] += float(pred.sum() + true.sum())

    scores = []
    per_class = {}

    for class_id, subtype in enumerate(ICH_CLASSES, start=1):
        denominator = denominators[class_id - 1]
        score = np.nan if denominator == 0 else 2.0 * intersections[class_id - 1] / denominator
        per_class[subtype] = score
        if np.isfinite(score):
            scores.append(score)

    return float(np.mean(scores)), per_class

## 14. Train the 3D model

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
best_path = MODELS_DIR / "ich_3d_resunet_v2_best.pth"
history_path = METRICS_DIR / "02_training_history.csv"

best_dice = -np.inf
best_epoch = None
epochs_without_improvement = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    n_batches = 0
    bar = tqdm(train_loader, desc=f"3D epoch {epoch + 1}/{EPOCHS}", leave=False)

    for volumes, targets, supervision, _, _ in bar:
        volumes = volumes.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        supervision = supervision.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            logits = model(volumes)
            loss, ce, dice = masked_loss(logits, targets, supervision)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.item())
        n_batches += 1
        bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / max(n_batches, 1)
    dev_mean_dice, dev_class_dice = evaluate_center_dice(model, dev_loader)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "dev_mean_dice": dev_mean_dice, **{f"dev_dice_{key}": value for key, value in dev_class_dice.items()}})
    pd.DataFrame(history).to_csv(history_path, index=False)

    marker = ""
    if dev_mean_dice > best_dice:
        best_dice = dev_mean_dice
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_path)
        marker = " <-- Best"
    else:
        epochs_without_improvement += 1

    print(f"Epoch {epoch + 1:02d} | loss {train_loss:.4f} | DEV center Dice {dev_mean_dice:.4f}{marker}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping after epoch {epoch + 1}.")
        break

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
model.eval()

print("Best epoch:", best_epoch)
print("Best DEV center Dice:", round(best_dice, 4))

## 15. Tune class-confidence thresholds on DEV center slices

In [ ]:
@torch.inference_mode()
def tune_thresholds(model, loader):
    intersections = {class_id: np.zeros(len(THRESHOLD_GRID), dtype=np.float64) for class_id in range(1, 6)}
    denominators = {class_id: np.zeros(len(THRESHOLD_GRID), dtype=np.float64) for class_id in range(1, 6)}

    for volumes, targets, supervision, _, _ in tqdm(loader, desc="Threshold tuning", leave=False):
        probabilities = torch.softmax(model(volumes.to(DEVICE))[:, :, CENTER_SLOT], dim=1).cpu()
        hard = torch.argmax(probabilities, dim=1)
        truth = targets[:, CENTER_SLOT]

        for class_id in range(1, 6):
            target = truth == class_id
            class_prob = probabilities[:, class_id]
            hard_class = hard == class_id

            for threshold_index, threshold in enumerate(THRESHOLD_GRID):
                pred = hard_class & (class_prob >= float(threshold))
                intersections[class_id][threshold_index] += float((pred & target).sum())
                denominators[class_id][threshold_index] += float(pred.sum() + target.sum())

    thresholds = {}
    rows = []

    for class_id, subtype in enumerate(ICH_CLASSES, start=1):
        dice = np.where(denominators[class_id] > 0, 2.0 * intersections[class_id] / denominators[class_id], 0.0)
        best_index = int(np.argmax(dice))
        thresholds[class_id] = float(THRESHOLD_GRID[best_index])
        rows.append({"class_id": class_id, "subtype": subtype, "threshold": thresholds[class_id], "dice": float(dice[best_index])})

    return thresholds, pd.DataFrame(rows)

thresholds, threshold_df = tune_thresholds(model, dev_loader)
threshold_df.to_csv(METRICS_DIR / "03_thresholds.csv", index=False)
display(threshold_df)

## 16. Full-series sliding-window 3D inference on DEV

In [ ]:
def derive_slice_depths_mm(series_table):
    positions = series_table["position_scalar"].to_numpy(dtype=np.float64)
    n = len(series_table)

    if n > 1 and np.all(np.isfinite(positions)):
        diffs = np.abs(np.diff(positions))
        positive = diffs[diffs > 1e-6]

        if len(positive):
            fallback = float(np.median(positive))
            diffs = np.where(diffs > 1e-6, diffs, fallback)
            depths = np.empty(n, dtype=np.float64)
            depths[0], depths[-1] = diffs[0], diffs[-1]
            if n > 2:
                depths[1:-1] = (diffs[:-1] + diffs[1:]) / 2.0
            return depths

    depths = []

    for row in series_table.itertuples(index=False):
        depth = float(row.spacing_between_slices)
        if not np.isfinite(depth) or depth <= 0:
            depth = float(row.slice_thickness)
        if not np.isfinite(depth) or depth <= 0:
            raise RuntimeError(f"Unable to determine slice depth for series {row.series_id}.")
        depths.append(depth)

    return np.asarray(depths, dtype=np.float64)

def inference_starts(n_slices):
    if n_slices <= PATCH_DEPTH:
        return [0]
    starts = list(range(0, n_slices - PATCH_DEPTH + 1, INFERENCE_STRIDE))
    last = n_slices - PATCH_DEPTH
    if starts[-1] != last:
        starts.append(last)
    return starts

@torch.inference_mode()
def predict_series_3d(series_id):
    group = series_rows[str(series_id)].copy()
    n_slices = len(group)
    probability_sum = torch.zeros((n_slices, 6, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)
    prediction_count = torch.zeros(n_slices, dtype=torch.float32)

    for start in inference_starts(n_slices):
        input_indices = [min(start + slot, n_slices - 1) for slot in range(PATCH_DEPTH)]
        volume = torch.stack([blood_window_tensor(load_hu_image(str(group.iloc[index]["dicom_path"]))) for index in input_indices]).unsqueeze(0).unsqueeze(0).to(DEVICE)
        probabilities = torch.softmax(model(volume)[0], dim=0).cpu()

        for slot in range(PATCH_DEPTH):
            original_index = start + slot
            if original_index >= n_slices:
                continue
            probability_sum[original_index] += probabilities[:, slot]
            prediction_count[original_index] += 1.0

    averaged = probability_sum / prediction_count[:, None, None, None].clamp_min(1.0)
    hard = torch.argmax(averaged, dim=1)
    final = torch.zeros_like(hard)

    for class_id in range(1, 6):
        final[(hard == class_id) & (averaged[:, class_id] >= thresholds[class_id])] = class_id

    depths = derive_slice_depths_mm(group)
    volumes = {subtype: 0.0 for subtype in ICH_CLASSES}
    slice_records = []

    for index in range(n_slices):
        row = group.iloc[index]
        scale_y = float(row["rows"]) / IMAGE_SIZE
        scale_x = float(row["cols"]) / IMAGE_SIZE
        voxel_volume_ml = float(row["row_spacing"]) * scale_y * float(row["col_spacing"]) * scale_x * float(depths[index]) / 1000.0
        slice_record = {"series_id": str(series_id), "slice_order": index, "sop_uid": str(row["sop_uid"])}

        for class_id, subtype in enumerate(ICH_CLASSES, start=1):
            pixels = int((final[index] == class_id).sum().item())
            volume = pixels * voxel_volume_ml
            volumes[subtype] += volume
            slice_record[f"pred_positive_{subtype}"] = int(pixels > 0)
            slice_record[f"pred_volume_{subtype}"] = float(volume)
            slice_record[f"max_probability_{subtype}"] = float(averaged[index, class_id].max().item())

        slice_records.append(slice_record)

    result = {"series_id": str(series_id), **{f"V_{subtype}": float(volumes[subtype]) for subtype in ICH_CLASSES}}
    return result, pd.DataFrame(slice_records)

series_predictions = []
slice_predictions = []
runtime_rows = []

for series_id in tqdm(dev_series_df["series_id"], desc="3D DEV full-series inference"):
    started = time.perf_counter()
    series_result, slice_result = predict_series_3d(series_id)
    runtime_rows.append({"series_id": series_id, "runtime_seconds": time.perf_counter() - started})
    series_predictions.append(series_result)
    slice_predictions.append(slice_result)

dev_pred_df = pd.DataFrame(series_predictions)
dev_slice_df = pd.concat(slice_predictions, ignore_index=True)
runtime_df = pd.DataFrame(runtime_rows)

dev_pred_df.to_csv(METRICS_DIR / "04_dev_series_predictions.csv", index=False)
dev_slice_df.to_csv(CACHE_DIR / "dev_slice_predictions.csv", index=False)
runtime_df.to_csv(METRICS_DIR / "05_dev_runtime.csv", index=False)

## 17. Series-level ICH metrics

In [ ]:
def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else np.nan,
        "FPR": float(fp / (tn + fp)) if tn + fp else np.nan,
    }

merged = dev_series_df.merge(dev_pred_df, on="series_id", suffixes=("_true", "_pred"))
true_total = merged[[f"{column}_true" for column in ICH_COLUMNS]].sum(axis=1)
pred_total = merged[[f"{column}_pred" for column in ICH_COLUMNS]].sum(axis=1)
presence = binary_metrics((true_total >= 0.1).astype(int), (pred_total >= 0.1).astype(int))

summary = {
    "best_DEV_center_Dice": best_dice,
    "Any_ICH_F1": presence["F1"],
    "Any_ICH_precision": presence["precision"],
    "Any_ICH_recall": presence["recall"],
    "Any_ICH_specificity": presence["specificity"],
    "Any_ICH_FPR": presence["FPR"],
    "Any_ICH_FP": presence["FP"],
    "Any_ICH_FN": presence["FN"],
    "total_ICH_MAE_mL": float(mean_absolute_error(true_total, pred_total)),
    "mean_runtime_seconds": float(runtime_df["runtime_seconds"].mean()),
}

for subtype in ICH_CLASSES:
    column = f"V_{subtype}"
    summary[f"{subtype}_MAE_mL"] = float(mean_absolute_error(merged[f"{column}_true"], merged[f"{column}_pred"]))
    summary[f"{subtype}_presence_F1"] = float(f1_score((merged[f"{column}_true"] >= 0.1).astype(int), (merged[f"{column}_pred"] >= 0.1).astype(int), zero_division=0))

summary_df = pd.DataFrame([summary])
summary_df.to_csv(METRICS_DIR / "06_dev_ich_summary.csv", index=False)
display(summary_df)

## 18. False-positive run analysis

In [ ]:
def extract_run_lengths(group, positive_column):
    flags = group.sort_values("slice_order")[positive_column].astype(bool).to_numpy()
    runs = []
    start = None

    for index in range(len(flags) + 1):
        active = index < len(flags) and flags[index]
        if active and start is None:
            start = index
        if not active and start is not None:
            runs.append(index - start)
            start = None

    return runs

negative_dev_series = set(dev_series_df.loc[dev_series_df[ICH_COLUMNS].sum(axis=1) < 0.1, "series_id"])
negative_slices = dev_slice_df[dev_slice_df["series_id"].isin(negative_dev_series)].copy()
run_rows = []

for subtype in ICH_CLASSES:
    lengths = []
    for _, group in negative_slices.groupby("series_id"):
        lengths.extend(extract_run_lengths(group, f"pred_positive_{subtype}"))

    run_rows.append({
        "subtype": subtype,
        "n_false_positive_runs": len(lengths),
        "median_run_length": float(np.median(lengths)) if lengths else np.nan,
        "single_slice_run_rate": float(np.mean(np.asarray(lengths) == 1)) if lengths else np.nan,
        "run_le_2_rate": float(np.mean(np.asarray(lengths) <= 2)) if lengths else np.nan,
    })

run_df = pd.DataFrame(run_rows)
run_df.to_csv(METRICS_DIR / "07_false_positive_runs.csv", index=False)
display(run_df)

## 19. Fixed three-slice continuity diagnostic

In [ ]:
def filter_subtype_runs(slice_df, subtype, min_run):
    work = slice_df.copy()
    output_column = f"filtered_volume_{subtype}"
    work[output_column] = 0.0

    for _, group in work.groupby("series_id", sort=False):
        group = group.sort_values("slice_order")
        flags = group[f"pred_positive_{subtype}"].astype(bool).to_numpy()
        keep = np.zeros(len(group), dtype=bool)
        start = None

        for index in range(len(flags) + 1):
            active = index < len(flags) and flags[index]
            if active and start is None:
                start = index
            if not active and start is not None:
                if index - start >= min_run:
                    keep[start:index] = True
                start = None

        work.loc[group.index, output_column] = np.where(keep, group[f"pred_volume_{subtype}"], 0.0)

    return work

filtered_slices = dev_slice_df.copy()
for subtype in ICH_CLASSES:
    filtered_slices = filter_subtype_runs(filtered_slices, subtype, FIXED_CONTINUITY_MIN_RUN)

filtered_series = filtered_slices.groupby("series_id")[[f"filtered_volume_{subtype}" for subtype in ICH_CLASSES]].sum().reset_index()
filtered_series = filtered_series.rename(columns={f"filtered_volume_{subtype}": f"V_{subtype}" for subtype in ICH_CLASSES})
filtered_merged = dev_series_df.merge(filtered_series, on="series_id", suffixes=("_true", "_pred"))
filtered_true_total = filtered_merged[[f"{column}_true" for column in ICH_COLUMNS]].sum(axis=1)
filtered_pred_total = filtered_merged[[f"{column}_pred" for column in ICH_COLUMNS]].sum(axis=1)
filtered_presence = binary_metrics((filtered_true_total >= 0.1).astype(int), (filtered_pred_total >= 0.1).astype(int))

continuity_df = pd.DataFrame([{
    "continuity_min_run": FIXED_CONTINUITY_MIN_RUN,
    "Any_ICH_F1": filtered_presence["F1"],
    "Any_ICH_precision": filtered_presence["precision"],
    "Any_ICH_recall": filtered_presence["recall"],
    "Any_ICH_specificity": filtered_presence["specificity"],
    "Any_ICH_FPR": filtered_presence["FPR"],
    "Any_ICH_FP": filtered_presence["FP"],
    "Any_ICH_FN": filtered_presence["FN"],
    "total_ICH_MAE_mL": float(mean_absolute_error(filtered_true_total, filtered_pred_total)),
}])

continuity_df.to_csv(METRICS_DIR / "08_fixed_continuity_diagnostic.csv", index=False)
display(continuity_df)

## 20. Save model and config

In [ ]:
state_path = MODELS_DIR / "ich_3d_resunet_v2_final.pth"
torch.save(model.state_dict(), state_path)

torchscript_path = MODELS_DIR / "ich_3d_resunet_v2_torchscript.pt"
with torch.inference_mode():
    traced = torch.jit.trace(model, torch.zeros(1, 1, PATCH_DEPTH, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))
traced.save(str(torchscript_path))

config = {
    "architecture": "custom_3d_resunet_v2",
    "base_channels": BASE_CHANNELS,
    "input_channels": 1,
    "patch_depth": PATCH_DEPTH,
    "image_size": IMAGE_SIZE,
    "inference_stride": INFERENCE_STRIDE,
    "blood_window": [BLOOD_WINDOW_CENTER, BLOOD_WINDOW_WIDTH],
    "class_names": ["Background"] + ICH_CLASSES,
    "class_confidence_thresholds": {str(key): float(value) for key, value in thresholds.items()},
    "expanded_true_negatives": True,
    "masked_unannotated_positive_series_slices": True,
    "best_validation_epoch": best_epoch,
    "best_DEV_center_Dice": best_dice,
    "locked_test_used": False,
}

config_path = MODELS_DIR / "ich_3d_resunet_v2_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

print("State dict:", state_path)
print("TorchScript:", torchscript_path)
print("Config:", config_path)

## 21. Direct answers

In [ ]:
direct_answers = pd.DataFrame([{
    "experiment": "3D ResUNet v2 Blood + expanded true negatives",
    "patch_depth": PATCH_DEPTH,
    "image_size": IMAGE_SIZE,
    "best_DEV_center_Dice": best_dice,
    "DEV_Any_ICH_F1": summary["Any_ICH_F1"],
    "DEV_Any_ICH_precision": summary["Any_ICH_precision"],
    "DEV_Any_ICH_recall": summary["Any_ICH_recall"],
    "DEV_Any_ICH_FPR": summary["Any_ICH_FPR"],
    "DEV_total_ICH_MAE_mL": summary["total_ICH_MAE_mL"],
    "DEV_mean_runtime_seconds": summary["mean_runtime_seconds"],
}])

direct_answers.to_csv(OUTPUT_ROOT / "00_DIRECT_ANSWERS.csv", index=False)
display(direct_answers)

print("Primary result:", OUTPUT_ROOT / "00_DIRECT_ANSWERS.csv")
print("ICH summary:", METRICS_DIR / "06_dev_ich_summary.csv")
print("False-positive runs:", METRICS_DIR / "07_false_positive_runs.csv")
print("Continuity diagnostic:", METRICS_DIR / "08_fixed_continuity_diagnostic.csv")

# Interpretation

Compare this model with:

- Notebook 17: 2D Blood + expanded true negatives
- Notebook 18: 2.5D Blood + expanded true negatives

The most important metrics are series-level Any-ICH FPR, recall, F1, volume MAE, false-positive run structure, and runtime. Pixel Dice is secondary to the series-level triage behavior.